# Colab Setup
Use the next few cells to clone PyCCAPT, install the Colab-only dependencies, and restart the runtime before opening the visualization workflow.


Start by cloning the PyCCAPT repository into the current Colab session.


In [ ]:
from pathlib import Path

repo_dir = Path('/content/pyccapt')
if repo_dir.exists():
    print(f'Using existing repository at {repo_dir}')
else:
    !git clone --branch main https://github.com/mmonajem/pyccapt.git /content/pyccapt


Install the notebook dependencies from the package extras instead of listing them one by one. This keeps the Colab setup aligned with the repository metadata.


In [ ]:
print('Dependencies will be installed from the repository extras after we enter /content/pyccapt.')


Move into the cloned repository before installing the package.


In [ ]:
%cd /content/pyccapt
%ls


Install PyCCAPT with the `calibration` extra so the tutorial dependencies come from the package definition. `gdown` is included here because the example data-download cells use it.


In [ ]:
%%capture
%pip install -U pip gdown
%pip install -e ".[calibration]"


# Data Visualization Workflow
Use this notebook to load an already processed dataset, optionally reload a range file, and open the interactive visualization tools.


In [ ]:
# Enable the custom widget manager for Colab and choose the best matplotlib backend available.
from google.colab import output
from IPython import get_ipython

output.enable_custom_widget_manager()

def _activate_matplotlib_backend():
    shell = get_ipython()
    for backend in ("widget", "notebook"):
        try:
            shell.run_line_magic("matplotlib", backend)
            print(f"Using matplotlib backend: {backend}")
            return backend
        except Exception as exc:
            print(f"Could not enable %matplotlib {backend}: {exc}")
    shell.run_line_magic("matplotlib", "inline")
    print("Falling back to %matplotlib inline. Plots still work, but interactive drawing tools may be limited in this runtime. Use the typed crop/index fields if needed.")
    return "inline"

COLAB_MATPLOTLIB_BACKEND = _activate_matplotlib_backend()
# Activate auto reload
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
# import libraries
from IPython.display import display
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

# Local module and scripts
from pyccapt.calibration.calibration import widgets as wd
from pyccapt.calibration.data_tools import data_tools
from pyccapt.calibration.tutorials.tutorials_helpers import helper_data_loader
from pyccapt.calibration.tutorials.tutorials_helpers import helper_visualization
from pyccapt.calibration.calibration import share_variables
from pyccapt.calibration.calibration import ion_selection


If HDF5 loading fails because `pytables` is missing on your Colab runtime, install it in a separate cell before continuing:

`%pip install tables`


Create the shared `variables` object first. The loaded dataset and range table are stored there for the visualization helpers.


In [ ]:
# Create the shared state container used by the Colab visualization workflow.
variables = share_variables.Variables()

Download the example dataset or replace the Google Drive file ID with your own processed file.


In [ ]:
! gdown https://drive.google.com/uc?id=1Jb8t7XlDGwuR13xl0HJMR-PNr9ACepw5
dataset_path = '/content/pyccapt/2382_Jan-10-2025_15-12_NiC9_Al_cropped_calibrated.h5'

In [ ]:
# Uncomment the lines below to upload your own processed dataset instead of using the example file.
# from google.colab import files
# uploaded = files.upload()
# dataset_path = next(iter(uploaded))


## Load Data And Optional Range Files
Set the instrument metadata, load the dataset, and optionally reload a previously saved range table.


Review the dataset settings before loading. The selected values control the derived arrays used by the visualization helpers.


In [ ]:
# create an object for selection of instrument specifications of the dataset
tdc, pulse_mode, flight_path_length, t0, max_mc, det_diam = wd.dataset_instrument_specification_selection()

# Display lists and comboboxes to selected instrument specifications
display(tdc, pulse_mode, flight_path_length, t0, max_mc)

In [ ]:
# Load the dataset with the selected settings and preview the dataset and range table.
helper_data_loader.load_data(dataset_path, max_mc.value, flight_path_length.value, pulse_mode.value, tdc.value, variables, processing_mode=False)
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)
display(variables.data)
display(variables.range_data)

If you already have a saved range file (`.h5`, `.rrng`, or `.rng`), load it here so the ion labels and colors are restored before visualization.


In [ ]:
# Optional example range file download. Uncomment these lines if you want to preload a saved range table.
# ! gdown <google-drive-file-id>
# range_path = '/content/pyccapt/example_range.h5'


In [ ]:
# Uncomment the lines below to upload your own saved range file.
# from google.colab import files
# uploaded = files.upload()
# range_path = next(iter(uploaded))


In [ ]:
# If a range file was chosen, load it into the shared state and preview it.
if 'range_path' in globals():
    variables.range_data = data_tools.read_range(range_path)
display(variables.range_data.style.applymap(ion_selection.display_color, subset=['color']))

Use the next cell to export the current range table once the ion list looks correct.


In [ ]:
# Export the current range table from Colab.
from google.colab import files
variables.range_data.to_hdf('range_' + variables.dataset_name + '.h5', key='df', mode='w')
variables.range_data.to_csv('range_' + variables.dataset_name + '.csv', encoding='utf-8', index=False, sep=';')
files.download('range_' + variables.dataset_name + '.h5')
files.download('range_' + variables.dataset_name + '.csv')

In [ ]:
variables.data

Use the next cell to export the current dataset in any additional formats you need.


In [ ]:
# Export the current dataset in the formats you need.
# By default the whole dataset is exported; change the indices below if you want a smaller subset.
export_name = variables.result_data_name
export_last_index = max(0, len(variables.data) - 1)
export_start_index = 0
export_end_index = export_last_index
data_tools.save_data(
    variables.data,
    variables,
    name=export_name,
    hdf=True,
    epos=False,
    pos=False,
    ato_6v=False,
    csv=False,
    start_index=export_start_index,
    end_index=export_end_index,
)
print(f'HDF5 export: {variables.resolve_result_data_file(export_name + ".h5")}')


## Visualization
The final cells refresh the extracted arrays and open the interactive visualization interface.

The `3D` and `Iso surface` panels now include optional Min-Max clustering controls. Turn clustering on, enter the ion or element labels for the precipitate of interest, and PyCCAPT will split that selected population into two precipitate segments in the reconstruction view.


In [ ]:
# Refresh the extracted arrays before launching the visualization helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

In [ ]:
# Open the Colab-compatible visualization helper.
# Because Colab does not support ipywidgets.Tab the same way as local Jupyter, use the dedicated Colab mode.
helper_visualization.call_visualization(variables, colab=True)